In [1]:
# GPU Allocation
# core = 3

# Which model, which dataset? 
model_choice, dataset_choice, glide_choice = "dino", "ham", True

In [2]:
%pip install -q \
    torch torchvision \
    transformers datasets accelerate evaluate \
    scikit-learn pillow ipywidgets ipykernel

import os
import sys
import gc

import numpy as np
import torch

from datasets import load_dataset
import evaluate

from torchvision.transforms import (
    CenterCrop, Compose, Normalize,
    RandomHorizontalFlip, RandomResizedCrop,
    Resize, ToTensor
)

from sklearn.metrics import accuracy_score

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)

Note: you may need to restart the kernel to use updated packages.


In [3]:
dataset_paths = {
    "ham": "/data/team01/ds340w/datasets/ham10000_split",
    "atlas_isic": "/data/team01/ds340w/datasets/Atlas_ISIC_split",
    "glide_ham": "/data/team01/ds340w/datasets/HAM_GLIDE_merged",
    "glide_atlas_isic": "/data/team01/ds340w/datasets/Atlas_ISIC_GLIDE_merged"
    
}


base_models = {
    "swin": "microsoft/swin-base-patch4-window7-224",
    "dino": "facebook/dinov2-base",
    "vit": "google/vit-base-patch16-224"
}

batch_sizes = {
    "swin": 16,
    "vit": 32,
    "dino": 32
}

In [4]:
# allocate the GPU
# os.environ["CUDA_VISIBLE_DEVICES"] = f"{core}"
# def clear_gpu():
#     torch.cuda.empty_cache()
#     gc.collect()
# clear_gpu()

In [5]:
# get base model
model_checkpoint = base_models[model_choice]

# load dataset for training
d_k = f"{"glide_" if glide_choice else ""}{dataset_choice}"
dataset = load_dataset("imagefolder", data_dir=dataset_paths[d_k])
metric = evaluate.load("accuracy")

# hash table
labels = dataset["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label


# transformations and preprocessing
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint)
normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
if "height" in image_processor.size:
    size = (image_processor.size["height"], image_processor.size["width"])
    crop_size = size
    max_size = None
elif "shortest_edge" in image_processor.size:
    size = image_processor.size["shortest_edge"]
    crop_size = (size, size)
    max_size = image_processor.size.get("longest_edge")

train_transforms = Compose(
        [
            RandomResizedCrop(crop_size),
            RandomHorizontalFlip(),
            ToTensor(),
            normalize,
        ]
    )

val_transforms = Compose(
        [
            Resize(size),
            CenterCrop(crop_size),
            ToTensor(),
            normalize,
        ]
    )

def preprocess_train(example_batch):
    """Apply train_transforms across a batch."""
    example_batch["pixel_values"] = [
        train_transforms(image.convert("RGB")) for image in example_batch["image"]
    ]
    return example_batch

def preprocess_val(example_batch):
    """Apply val_transforms across a batch."""
    example_batch["pixel_values"] = [val_transforms(image.convert("RGB")) for image in example_batch["image"]]
    return example_batch

train_ds = dataset["train"]
validation = load_dataset("imagefolder", data_dir=f"{dataset_paths[dataset_choice]}")
val_ds = validation["test"]

# apply the transformations
train_ds.set_transform(preprocess_train)
val_ds.set_transform(preprocess_val)

# create the model object
model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes = True, 
)
model_name = model_checkpoint.split("/")[-1]

# training arguments, 10 iterations, 32 batch, and training step
args = TrainingArguments(
    f"{model_choice}.{dataset_choice}.{"glide" if glide_choice else "original"}-finetuned-SkinDisease", # save state filename
    remove_unused_columns=False,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_sizes[model_choice],
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=batch_sizes[model_choice],
    num_train_epochs=10,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=True,
)

Resolving data files:   0%|          | 0/7470 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/7010 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2004 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1001 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dinov2ForImageClassification LOAD REPORT from: facebook/dinov2-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [6]:
print(d_k) # sanity check!

glide_ham


In [7]:
def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

In [8]:
# training the model
trainer = Trainer(
        model,
        args,
        train_dataset=train_ds, # training set
        eval_dataset=val_ds, # validation set
        processing_class=image_processor,
        compute_metrics=compute_metrics,
        data_collator=collate_fn,
        callbacks = []
    )

In [ ]:
# run training
train_results = trainer.train()
trainer.save_model()
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)
trainer.save_state()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.773210,0.921210,0.705295
2,0.656500,0.659286,0.728272
3,0.576514,0.636484,0.791209
4,0.474894,0.403978,0.843157


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# test dataset loading
test_images = load_dataset("imagefolder", data_dir=f"{dataset_paths[dataset_choice]}/test")

# Preprocess the test images
test_images.set_transform(preprocess_val)  # Use the same validation transforms
print("transforms on test data")

# Create a test dataset
test_ds = test_images["train"]  # Use the "train" split for the test data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Good practice: set model to evaluation mode
print(f"Using device: {device}")

# Initialize lists to store predicted and actual labels
predicted_labels = []
actual_labels = []

# Iterate through the test dataset and make predictions
print("test predictions started")
for example in test_ds:
    image = example["image"]
    
    # 2. Process the image (This creates tensors on the CPU by default)
    encoding = image_processor(image.convert("RGB"), return_tensors="pt")

    # 3. MOVE THE IMAGES TO GPU
    # We loop through the dictionary and move each tensor to the GPU
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        # 4. Now both the model and the encoding are on the same device
        outputs = model(**encoding)
        logits = outputs.logits

    # 5. Bring the result back to CPU to extract the number
    predicted_class_idx = logits.argmax(-1).item()
    predicted_labels.append(predicted_class_idx)
    actual_labels.append(example["label"])

# Calculate accuracy
accuracy = accuracy_score(actual_labels, predicted_labels)
print(f"Test Accuracy: {accuracy:.4f}")